# GOLD LAYER

## Ventas Agregadas

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW agg_sales_view AS
SELECT
  Date_Year,
  Date_Month,
  Product_Category,
  SUM(Total_Amount) AS total_sales,
  SUM(Quantity) AS total_quantity,
  COUNT(DISTINCT Customer_ID) AS total_customers
FROM retail_sales.silver.raw_dedup_view_month_year
WHERE Processed = FALSE
GROUP BY
  Date_Year,
  Date_Month,
  Product_Category;

MERGE INTO retail_sales.gold.agg_sales A   
USING agg_sales_view B
ON A.Date_Year = B.Date_Year AND A.Date_Month = B.Date_Month AND A.Product_Category = B.Product_Category
WHEN MATCHED THEN UPDATE SET
    A.total_sales = B.total_sales,
    A.total_quantity = B.total_quantity
WHEN NOT MATCHED THEN INSERT *

## CLIENT SEGMENTATION

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW agg_client_view AS
SELECT
  Date_Year,
  Date_Month,
  Gender,
  Age,
  COUNT(Transaction_ID) AS total_transactions,
  SUM(Total_Amount) AS total_spent,
  AVG(Total_Amount) AS avg_ticket
FROM retail_sales.silver.raw_dedup_view_month_year
WHERE Processed = FALSE
GROUP BY
  Date_Year,
  Date_Month,
  Age,
  Gender
  ;

MERGE INTO retail_sales.gold.agg_client A 
USING agg_client_view B
ON A.Date_Year = B.Date_Year AND A.Date_Month = B.Date_Month AND A.Age = B.Age AND A.Gender = B.Gender
WHEN MATCHED THEN UPDATE SET
  A.total_transactions = B.total_transactions,
  A.total_spent = B.total_spent,
  A.avg_ticket = B.avg_ticket
WHEN NOT MATCHED THEN INSERT *

Update the processed flag to true in the retail_sales.silver.raw_dedup_view_month_year table

In [0]:
%sql
UPDATE retail_sales.silver.raw_dedup_view_month_year
SET Processed = TRUE
WHERE Processed = FALSE